# Compare classifier ensembles

This notebook loads the best ConvNeXt Tiny, ViT Tiny, Swin Tiny, and EfficientNetV2-S checkpoints selected by their training notebooks. It compares hard voting, equal probability voting, globally weighted probability voting, and model-by-class weighted probability voting. Weights are derived only from validation data. Complete validation and test results are saved for every ensemble method, while the best validation macro-F1 method is still recorded as the validation-selected method. Each model retains its own saved context radius, input size, and training-derived normalization statistics.

Missing or stale split/checkpoint provenance is rejected: rerun the BS split and training notebooks first. `timing_summary.csv` reports each method with full component prediction costs, combination, and validation-only weight fitting. Validation setup plus test is distinct from test prediction latency. Existing outputs may predate this policy.


In [ ]:
from pathlib import Path
import json
import hashlib
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.beach_dataset import read_class_name_mapping, read_split_csv
from Source.ensemble import (
    average_probabilities,
    class_weighted_average_probabilities,
    hard_voting_predictions,
    load_model_artifact,
    macro_f1_weights,
    per_class_f1_weights,
    predict_artifact_on_split,
    validate_artifact_compatibility,
    weighted_average_probabilities,
)
from Source.training import classification_metrics, get_device
TEST_DATES = {"20180812", "20180915", "20220910"}
SPLIT_POLICY = "bs_date_holdout_20180812_20180915_20220910_v1"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def current_split_provenance():
    return {
        "policy": SPLIT_POLICY,
        "test_dates": sorted(TEST_DATES),
        "validation_fraction_of_remainder": 15 / 85,
        "csv_sha256": {
            name: file_sha256(PREPROCESSED_DIR / f"bs_{name}_split.csv")
            for name in ("train", "validation", "test")
        },
    }


def validate_bs_splits(train, validation, test):
    frames = {"train": train, "validation": validation, "test": test}
    for name, frame in frames.items():
        if frame.empty or not frame["split"].eq(name).all():
            raise ValueError(f"Invalid or empty BS {name} split; regenerate BS splits.")
        held_out = frame["date"].astype(str).isin(TEST_DATES)
        if not (held_out.all() if name == "test" else not held_out.any()):
            raise ValueError("BS date holdout violated; rerun bs_create_dataset_splits.")
    if set(test["date"].astype(str)) != TEST_DATES:
        raise ValueError("BS test split must contain all three held-out dates.")
    combined = pd.concat(list(frames.values()), ignore_index=True)
    if combined.duplicated(["date", "cell_id"]).any():
        raise ValueError("Duplicate or overlapping BS split cells.")


def synchronize_timing():
    # Grid search can use every CUDA device, not just the selected device.
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            torch.cuda.synchronize(index)
    if torch.backends.mps.is_available():
        torch.mps.synchronize()


def timed_call(function, *args, **kwargs):
    synchronize_timing()
    started = perf_counter()
    result = function(*args, **kwargs)
    synchronize_timing()
    return result, perf_counter() - started


In [ ]:
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
CLASS_MAPPING_PATH = PROJECT_ROOT / "Data" / "Raw" / "Classes.xlsx"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
OUTPUT_DIR = PROJECT_ROOT / "Results" / "bs_ensemble"

VALIDATION_SPLIT = PREPROCESSED_DIR / "bs_validation_split.csv"
TEST_SPLIT = PREPROCESSED_DIR / "bs_test_split.csv"

MODEL_RESULT_DIRS = {
    "convnext_tiny": PROJECT_ROOT / "Results" / "bs_convnext_tiny",
    "vit_tiny": PROJECT_ROOT / "Results" / "bs_vit_tiny",
    "swin_tiny": PROJECT_ROOT / "Results" / "bs_swin_tiny",
    "efficientnetv2_s": PROJECT_ROOT / "Results" / "bs_efficientnetv2_s",
}

BATCH_SIZE = 32
NUM_WORKERS = 8

print(f"Project root:   {PROJECT_ROOT}", flush=True)
print(f"Cell data root: {CELL_DATA_ROOT}", flush=True)
print(f"Output dir:     {OUTPUT_DIR}", flush=True)

In [ ]:
print("Loading splits and best-model artifacts...", flush=True)
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Cell data directory not found: {CELL_DATA_ROOT}")

validation_df = read_split_csv(VALIDATION_SPLIT)
test_df = read_split_csv(TEST_SPLIT)
train_df = read_split_csv(PREPROCESSED_DIR / "bs_train_split.csv")
validate_bs_splits(train_df, validation_df, test_df)
split_provenance = current_split_provenance()
artifacts = [
    load_model_artifact(name, result_dir)
    for name, result_dir in MODEL_RESULT_DIRS.items()
]
validate_artifact_compatibility(artifacts)
for artifact in artifacts:
    if artifact.metadata.get("split_provenance") != split_provenance:
        raise ValueError(f"{artifact.name}: missing/stale BS split provenance; retrain on current splits.")
    if artifact.metadata.get("checkpoint_sha256") != file_sha256(artifact.checkpoint_path):
        raise ValueError(f"{artifact.name}: checkpoint does not match its provenance; retrain.")

reference_metadata = artifacts[0].metadata
class_to_idx = {
    str(code): int(index)
    for code, index in reference_metadata["class_to_idx"].items()
}
idx_to_class_code = {
    int(index): str(code)
    for index, code in reference_metadata["idx_to_class_code"].items()
}
expected_classes = set(class_to_idx)
for split_name, split_df in [("validation", validation_df), ("test", test_df)]:
    split_classes = set(split_df["class"].astype(str))
    if not split_classes.issubset(expected_classes):
        raise ValueError(
            f"{split_name} split contains classes absent from model metadata: "
            f"{sorted(split_classes - expected_classes)}"
        )

class_name_by_code = read_class_name_mapping(CLASS_MAPPING_PATH, expected_classes)
idx_to_class_name = {
    index: f"{code}: {class_name_by_code.get(code, f'class_{code}')}"
    for index, code in idx_to_class_code.items()
}

artifact_table = pd.DataFrame(
    [
        {
            "model": artifact.name,
            "architecture": artifact.metadata.get("model_name", artifact.metadata["model_type"]),
            "context_radius": artifact.metadata["context_radius"],
            "image_size": artifact.metadata["image_size"],
            "checkpoint": str(artifact.checkpoint_path),
        }
        for artifact in artifacts
    ]
)
print(f"Validation rows: {len(validation_df):,}", flush=True)
print(f"Test rows:       {len(test_df):,}", flush=True)
display(artifact_table)

In [ ]:
def metric_summary(method_name, y_true, y_pred):
    return {
        "method": method_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


print("Generating validation probabilities for ensemble search...", flush=True)
device = get_device()
validation_probabilities = {}
validation_component_seconds = {}
validation_y_true = None

for artifact in artifacts:
    print(f"Validation inference: {artifact.name}", flush=True)
    (current_y_true, current_probabilities), prediction_seconds = timed_call(
        predict_artifact_on_split,
        artifact,
        validation_df,
        CELL_DATA_ROOT,
        device=device,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    if validation_y_true is None:
        validation_y_true = current_y_true
    elif not np.array_equal(validation_y_true, current_y_true):
        raise RuntimeError(f"Validation target order differs for {artifact.name}.")
    validation_probabilities[artifact.name] = current_probabilities
    validation_component_seconds[artifact.name] = prediction_seconds

(validation_model_macro_f1, global_model_weights), global_weight_fit_seconds = timed_call(
    macro_f1_weights,
    validation_y_true,
    validation_probabilities,
)
(validation_class_f1, model_class_weights), class_weight_fit_seconds = timed_call(
    per_class_f1_weights,
    validation_y_true,
    validation_probabilities,
)

def combine_candidates(probabilities_by_model):
    operations = {
        "hard_voting": lambda: hard_voting_predictions(probabilities_by_model),
        "probability_voting": lambda: average_probabilities(probabilities_by_model),
        "weighted_probability_per_model": lambda: weighted_average_probabilities(
            probabilities_by_model, global_model_weights
        ),
        "weighted_probability_per_model_class": lambda: class_weighted_average_probabilities(
            probabilities_by_model, model_class_weights
        ),
    }
    probabilities, predictions, seconds = {}, {}, {}
    for method, operation in operations.items():
        def combine_and_predict():
            result = operation()
            if method == "hard_voting":
                return result
            return result.argmax(axis=1), result
        (predictions[method], probabilities[method]), seconds[method] = timed_call(
            combine_and_predict
        )
    return probabilities, predictions, seconds


(validation_candidate_probabilities, validation_candidate_predictions,
 validation_combination_seconds) = combine_candidates(validation_probabilities)

ensemble_method_order = [
    "hard_voting",
    "probability_voting",
    "weighted_probability_per_model",
    "weighted_probability_per_model_class",
]
validation_search_df = pd.DataFrame(
    [
        metric_summary(
            method,
            validation_y_true,
            validation_candidate_predictions[method],
        )
        for method in ensemble_method_order
    ]
)
best_ensemble_method = max(
    ensemble_method_order,
    key=lambda method: float(
        validation_search_df.loc[
            validation_search_df["method"] == method,
            "macro_f1",
        ].iloc[0]
    ),
)
validation_search_df["selected"] = validation_search_df["method"].eq(best_ensemble_method)
validation_ensemble_probability = validation_candidate_probabilities[best_ensemble_method]
validation_ensemble_pred = validation_candidate_predictions[best_ensemble_method]

global_weight_table = pd.DataFrame(
    [
        {
            "model": name,
            "validation_macro_f1": validation_model_macro_f1[name],
            "ensemble_weight": global_model_weights[name],
        }
        for name in validation_probabilities
    ]
).sort_values("validation_macro_f1", ascending=False)

class_weight_table = pd.DataFrame(
    [
        {
            "model": name,
            "class_index": class_index,
            "class_code": idx_to_class_code[class_index],
            "class_name": class_name_by_code.get(
                idx_to_class_code[class_index],
                f"class_{idx_to_class_code[class_index]}",
            ),
            "validation_class_f1": validation_class_f1[name][class_index],
            "ensemble_weight": model_class_weights[name][class_index],
        }
        for class_index in range(len(idx_to_class_code))
        for name in validation_probabilities
    ]
).sort_values(["class_index", "ensemble_weight"], ascending=[True, False])

validation_reports = {}
validation_confusions = {}
for method in ensemble_method_order:
    report, confusion = classification_metrics(
        validation_y_true,
        validation_candidate_predictions[method],
        idx_to_class_name,
    )
    validation_reports[method] = report
    validation_confusions[method] = confusion

print(f"Selected ensemble method: {best_ensemble_method}", flush=True)
display(validation_search_df.sort_values("macro_f1", ascending=False))
display(global_weight_table)
display(class_weight_table)

In [ ]:
print("Ensemble definitions are frozen. Generating test predictions for all methods...", flush=True)
if current_split_provenance() != split_provenance:
    raise RuntimeError("BS splits changed during ensemble evaluation; restart evaluation.")
test_probabilities = {}
test_component_seconds = {}
test_y_true = None

for artifact in artifacts:
    print(f"Test inference: {artifact.name}", flush=True)
    (current_y_true, current_probabilities), prediction_seconds = timed_call(
        predict_artifact_on_split,
        artifact,
        test_df,
        CELL_DATA_ROOT,
        device=device,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    if test_y_true is None:
        test_y_true = current_y_true
    elif not np.array_equal(test_y_true, current_y_true):
        raise RuntimeError(f"Test target order differs for {artifact.name}.")
    test_probabilities[artifact.name] = current_probabilities
    test_component_seconds[artifact.name] = prediction_seconds

(test_candidate_probabilities, test_candidate_predictions,
 test_combination_seconds) = combine_candidates(test_probabilities)

test_ensemble_probability = test_candidate_probabilities[best_ensemble_method]
test_ensemble_pred = test_candidate_predictions[best_ensemble_method]
print("Finished all test inference and ensemble predictions.", flush=True)

In [ ]:
print("Computing and saving individual-model and all-ensemble metrics...", flush=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_rows = []
individual_predictions = {}

for name, probabilities in test_probabilities.items():
    predictions = probabilities.argmax(axis=1)
    individual_predictions[name] = predictions
    row = metric_summary(name, test_y_true, predictions)
    row["model"] = row.pop("method")
    row["result_type"] = "individual_model"
    comparison_rows.append(row)
    model_report, model_confusion = classification_metrics(
        test_y_true,
        predictions,
        idx_to_class_name,
    )
    model_report.to_csv(OUTPUT_DIR / f"test_{name}_classification_report.csv")
    model_confusion.to_csv(OUTPUT_DIR / f"test_{name}_confusion_matrix.csv")

test_ensemble_reports = {}
test_ensemble_confusions = {}
for method in ensemble_method_order:
    predictions = test_candidate_predictions[method]
    row = metric_summary(f"ensemble_{method}", test_y_true, predictions)
    row["model"] = row.pop("method")
    row["result_type"] = "ensemble"
    row["validation_selected"] = method == best_ensemble_method
    comparison_rows.append(row)
    report, confusion = classification_metrics(
        test_y_true,
        predictions,
        idx_to_class_name,
    )
    test_ensemble_reports[method] = report
    test_ensemble_confusions[method] = confusion
    report.to_csv(OUTPUT_DIR / f"test_{method}_classification_report.csv")
    confusion.to_csv(OUTPUT_DIR / f"test_{method}_confusion_matrix.csv")

comparison_df = pd.DataFrame(comparison_rows).sort_values("macro_f1", ascending=False)
ensemble_report = test_ensemble_reports[best_ensemble_method]
ensemble_confusion = test_ensemble_confusions[best_ensemble_method]

validation_search_df.to_csv(OUTPUT_DIR / "validation_ensemble_search.csv", index=False)
global_weight_table.to_csv(OUTPUT_DIR / "validation_model_weights.csv", index=False)
class_weight_table.to_csv(OUTPUT_DIR / "validation_model_class_weights.csv", index=False)
for method in ensemble_method_order:
    validation_reports[method].to_csv(
        OUTPUT_DIR / f"validation_{method}_classification_report.csv"
    )
    validation_confusions[method].to_csv(
        OUTPUT_DIR / f"validation_{method}_confusion_matrix.csv"
    )

validation_reports[best_ensemble_method].to_csv(
    OUTPUT_DIR / "validation_ensemble_classification_report.csv"
)
validation_confusions[best_ensemble_method].to_csv(
    OUTPUT_DIR / "validation_ensemble_confusion_matrix.csv"
)
comparison_df.to_csv(OUTPUT_DIR / "test_model_comparison.csv", index=False)
ensemble_report.to_csv(OUTPUT_DIR / "test_ensemble_classification_report.csv")
ensemble_confusion.to_csv(OUTPUT_DIR / "test_ensemble_confusion_matrix.csv")

display(comparison_df)
display(ensemble_report)
# Components are measured once and reused, but each method is charged their sum.
# Weight fitting is validation-only, never included as test-time fitting.
weight_fit_seconds = {
    "hard_voting": 0.0,
    "probability_voting": 0.0,
    "weighted_probability_per_model": global_weight_fit_seconds,
    "weighted_probability_per_model_class": class_weight_fit_seconds,
}
component_training_times = {}
for artifact in artifacts:
    component_timing = pd.read_csv(artifact.result_dir / "timing_summary.csv").iloc[0]
    component_training_times[artifact.name] = float(component_timing["grid_training_work_seconds_sum"])
timing_rows = []
for name in test_probabilities:
    timing_rows.append({
        "method": name,
        "training_seconds": component_training_times[name],
        "result_type": "individual_model",
        "validation_component_prediction_seconds_sum": validation_component_seconds[name],
        "test_component_prediction_seconds_sum": test_component_seconds[name],
        "validation_weight_fitting_seconds": 0.0,
        "validation_combination_seconds": 0.0,
        "test_combination_seconds": 0.0,
        "test_prediction_seconds_including_components": test_component_seconds[name],
        "validation_setup_plus_test_seconds": validation_component_seconds[name] + test_component_seconds[name],
    })
for method in ensemble_method_order:
    validation_cost = sum(validation_component_seconds.values())
    test_cost = sum(test_component_seconds.values())
    timing_rows.append({
        "method": method,
        "training_seconds": sum(component_training_times.values()) + weight_fit_seconds[method],
        "result_type": "ensemble",
        "validation_component_prediction_seconds_sum": validation_cost,
        "test_component_prediction_seconds_sum": test_cost,
        "validation_weight_fitting_seconds": weight_fit_seconds[method],
        "validation_combination_seconds": validation_combination_seconds[method],
        "test_combination_seconds": test_combination_seconds[method],
        "test_prediction_seconds_including_components": test_cost + test_combination_seconds[method],
        "validation_setup_plus_test_seconds": (
            validation_cost + weight_fit_seconds[method] + validation_combination_seconds[method]
            + test_cost + test_combination_seconds[method]
        ),
    })
timing_df = pd.DataFrame(timing_rows)
timing_df["timing_scope"] = (
    "synchronized wall clock; components include model loading and data loading; "
    "combination includes argmax; totals are attributed per method, not summed across methods; "
    "validation setup includes prediction, weight fitting and combination; "
    "training_seconds sums component grid training work plus weight fitting; excludes metrics and artifact writes"
)
timing_df.to_csv(OUTPUT_DIR / "timing_summary.csv", index=False)
display(timing_df)

quality_with_timing = comparison_df.copy()
quality_with_timing["method"] = quality_with_timing["model"].str.removeprefix("ensemble_")
quality_with_timing = quality_with_timing.merge(timing_df, on=["method", "result_type"], validate="one_to_one")
quality_with_timing.to_csv(OUTPUT_DIR / "test_metrics_and_timing.csv", index=False)
display(quality_with_timing)

In [ ]:
print("Saving predictions for all ensemble methods and the ensemble definition...", flush=True)
prediction_df = test_df.reset_index(drop=True).copy()
prediction_df["true_class_code"] = [idx_to_class_code[int(index)] for index in test_y_true]

for name, predictions in individual_predictions.items():
    prediction_df[f"{name}_predicted_class_code"] = [
        idx_to_class_code[int(index)] for index in predictions
    ]

prediction_df["validation_selected_ensemble_method"] = best_ensemble_method
for method in ensemble_method_order:
    predictions = test_candidate_predictions[method]
    probabilities = test_candidate_probabilities[method]
    prediction_df[f"{method}_predicted_class_code"] = [
        idx_to_class_code[int(index)] for index in predictions
    ]
    prediction_df[f"{method}_predicted_class_name"] = [
        class_name_by_code.get(code, f"class_{code}")
        for code in prediction_df[f"{method}_predicted_class_code"]
    ]
    for class_index, class_code in idx_to_class_code.items():
        prediction_df[
            f"{method}_probability_class_{class_code}"
        ] = probabilities[:, class_index]

prediction_path = OUTPUT_DIR / "test_predictions.csv"
prediction_df.to_csv(prediction_path, index=False)

ensemble_definition = {
    "split_provenance": split_provenance,
    "selection_metric": "validation_macro_f1",
    "candidate_methods": ensemble_method_order,
    "validation_selected_method": best_ensemble_method,
    "validation_search": validation_search_df.to_dict(orient="records"),
    "global_model_weights": {
        name: float(weight) for name, weight in global_model_weights.items()
    },
    "validation_model_macro_f1": {
        name: float(score) for name, score in validation_model_macro_f1.items()
    },
    "weights_by_model_and_class": {
        name: {
            idx_to_class_code[class_index]: float(weight)
            for class_index, weight in enumerate(weights)
        }
        for name, weights in model_class_weights.items()
    },
    "validation_f1_by_model_and_class": {
        name: {
            idx_to_class_code[class_index]: float(score)
            for class_index, score in enumerate(scores)
        }
        for name, scores in validation_class_f1.items()
    },
    "class_to_idx": class_to_idx,
    "idx_to_class_code": idx_to_class_code,
    "models": {
        artifact.name: {
            "result_dir": str(artifact.result_dir),
            "checkpoint_path": str(artifact.checkpoint_path),
            "metadata_path": str(artifact.result_dir / "inference_metadata.json"),
            "context_radius": int(artifact.metadata["context_radius"]),
            "image_size": int(artifact.metadata["image_size"]),
        }
        for artifact in artifacts
    },
}
definition_path = OUTPUT_DIR / "ensemble_definition.json"
with definition_path.open("w", encoding="utf-8") as file:
    json.dump(ensemble_definition, file, indent=2)

print(f"Validation-selected method: {best_ensemble_method}")
print(f"Saved ensemble definition: {definition_path}")
print(f"Saved all test predictions: {prediction_path}")
prediction_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_comparison = comparison_df.sort_values("macro_f1")
bar_colors = [
    "seagreen" if result_type == "ensemble" else "steelblue"
    for result_type in plot_comparison["result_type"]
]
axes[0].barh(plot_comparison["model"], plot_comparison["macro_f1"], color=bar_colors)
axes[0].set_xlabel("Test macro-F1")
axes[0].set_xlim(0, 1)
axes[0].set_title("Individual models and all ensemble methods")

image = axes[1].imshow(ensemble_confusion.to_numpy(), cmap="Blues")
axes[1].set_xticks(range(len(ensemble_confusion.columns)))
axes[1].set_yticks(range(len(ensemble_confusion.index)))
axes[1].set_xticklabels(ensemble_confusion.columns, rotation=45, ha="right")
axes[1].set_yticklabels(ensemble_confusion.index)
axes[1].set_xlabel("Predicted class")
axes[1].set_ylabel("True class")
axes[1].set_title(f"Validation-selected ensemble: {best_ensemble_method}")
fig.colorbar(image, ax=axes[1])

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_ensemble_comparison.png", dpi=150)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 13))
for ax, method in zip(axes.flat, ensemble_method_order):
    confusion = test_ensemble_confusions[method]
    image = ax.imshow(confusion.to_numpy(), cmap="Blues")
    ax.set_xticks(range(len(confusion.columns)))
    ax.set_yticks(range(len(confusion.index)))
    ax.set_xticklabels(confusion.columns, rotation=45, ha="right")
    ax.set_yticklabels(confusion.index)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(method)
    fig.colorbar(image, ax=ax)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_all_ensemble_confusion_matrices.png", dpi=150)
plt.show()